<a href="https://colab.research.google.com/github/flipiwolker-alt/cv-video-analytics/blob/main/notebooks/PZ_3_OCR_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ПЗ 3 — OCR субтитров из кадров

Читаем кадры из Drive, вырезаем нижнюю полосу с субтитрами, распознаём текст через EasyOCR.

In [1]:
!pip install easyocr opencv-python-headless pandas tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 16.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров: {len(frames)}')

Mounted at /content/drive
кадров: 438


In [3]:
import cv2
import easyocr
import pandas as pd
from tqdm.notebook import tqdm

reader = easyocr.Reader(['ru', 'en'], gpu=True)

SUBTITLE_FRACTION = 0.15  # нижние 15% кадра

rows = []

for fname in tqdm(frames, desc='ocr'):
    img = cv2.imread(f'{FRAMES_DIR}/{fname}')
    if img is None:
        continue
    h = img.shape[0]
    crop = img[int(h * (1 - SUBTITLE_FRACTION)):, :]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    for _, text, prob in reader.readtext(gray):
        text = text.strip()
        if text and prob > 0.4:
            rows.append({'frame': fname, 'text': text, 'conf': round(prob, 3)})

df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/ocr_results.csv', index=False, encoding='utf-8-sig')
print(f'распознано строк: {len(df)}')

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.1% Complete

ocr:   0%|          | 0/438 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


распознано строк: 761


In [5]:
from IPython.display import display
import pandas as pd

df_unique = pd.DataFrame({'текст': df['text'].drop_duplicates().reset_index(drop=True)})

print(f'уникальных фраз: {len(df_unique)}')
display(df_unique.head(20))

уникальных фраз: 79


,текст
0,We're no strangers to love
1,Любовь нам не чужда
2,Were no strangers to love
3,You know the rules and so do |
4,Ты знаешь ee правила; как и я
5,"Ты знаешь ee правила, как и я"
6,You knowthe rules and so do |
7,Afull commitment's what Im thinking of
8,настроен серьезно
9,Afull commitment's what I'm
